# Open-Source CMOS PTAT Temperature Sensor with Digital Calibration

**ISSCC 2027 Code-a-Chip**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sscs-ose/sscs-ose-code-a-chip.github.io/blob/main/ISSCC27/submitted_notebooks/ptat_temp_sensor/PTAT_Temperature_Sensor_Code_a_Chip.ipynb)

This notebook is the primary reproducible artifact for a SKY130 PTAT temperature
sensor study. It separates analytical, retained real-PDK, synthetic, and
unavailable evidence so simulation results are not presented as silicon
measurements.

The core idea uses the weak-inversion relation ΔVGS ≈ n·kT/q·ln(N), with an
effective current-density ratio N=8, followed by digital calibration.

**Team member / submission representative:** Omar Islam ([@3more102](https://github.com/3more102))

**GitHub submission contact:** [@3more102](https://github.com/3more102)


## Submission identity and verified tool versions

- Submission contact: **@3more102** (GitHub repository owner/contact).
- Primary artifact: this Jupyter notebook, released under Apache-2.0 with the project files.
- Retained transistor evidence: **ngspice 46**.
- Retained open PDK provenance: **open_pdks/SKY130 `12df12e2e74145e31c5a13de02f9a1e176b56e67`**.
- Upstream Code-a-Chip notebook CI uses **Python 3.10**; repository CI also verifies on **Python 3.12**.

The retained run-221 manifest is the source of truth for the SPICE/PDK versions; the CI Python version describes current reproducibility verification, not the historical transistor run.


## Evidence contract

The retained real-PDK evidence was generated with ngspice and SKY130/open_pdks.
The repository preserves raw CSV data and a machine-readable manifest.

The release calibration is fixed five-point PWL at **−40, −20, 0, 50, 125 °C**.
Endpoint two-point calibration is included as a disclosed negative result because
it does not meet the internal 0.5 °C target.

## Innovation and contribution

This project does **not** claim a new PTAT device topology. Its contribution is an
evidence-driven open-source design-closure workflow that makes calibration,
readout, mismatch, and claim boundaries reproducible rather than implicit.

The main technical contributions are:

- it preserves the negative result that endpoint two-point calibration misses the
  internal 0.5 °C target, then closes the deterministic accuracy gap with a fixed
  five-point piecewise-linear calibration;
- it verifies that calibration on an actual 5 °C SKY130 TT/FF/SS transistor grid,
  where the worst sampled error is **0.472 °C**;
- it separates deterministic corner behavior from real local-device mismatch: the
  retained 100-seed `tt_mm` run is reported even though it misses the statistical
  targets, preventing calibration accuracy from being mistaken for mismatch yield;
- it carries the sensor result through an explicit 12-bit behavioral readout budget
  while keeping ADC and whole-chip power outside the transistor-level claim; and
- it retains raw evidence, provenance, automated audits, and reproducible scripts so
  every promoted result has a traceable evidence class.

The intended innovation is therefore the **auditable design methodology and design
closure**, not an unsupported claim of a new sensor topology.


In [ ]:
from pathlib import Path
import json
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_REL = Path(
    "ISSCC27", "submitted_notebooks", "ptat_temp_sensor"
)
ROOT = Path.cwd()

if not (ROOT / "results").is_dir():
    checkout_project = ROOT / PROJECT_REL
    if checkout_project.is_dir():
        ROOT = checkout_project
    else:
        upstream = ROOT / "sscs-ose-code-a-chip.github.io"
        if not upstream.is_dir():
            subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--filter=blob:none",
                    "--sparse",
                    "https://github.com/sscs-ose/"
                    "sscs-ose-code-a-chip.github.io.git",
                    str(upstream),
                ],
                check=True,
            )
        project_in_upstream = upstream / PROJECT_REL
        if not project_in_upstream.is_dir():
            subprocess.run(
                [
                    "git",
                    "-C",
                    str(upstream),
                    "sparse-checkout",
                    "set",
                    PROJECT_REL.as_posix(),
                ],
                check=True,
            )
        ROOT = project_in_upstream

if not (ROOT / "results").is_dir():
    raise FileNotFoundError(
        "Project supporting files were not found. "
        "Run from a repository checkout containing "
        f"{PROJECT_REL}, or use the Colab badge after the "
        "submission has been merged upstream."
    )

design = json.loads((ROOT / "design_requirements.json").read_text())
manifest = json.loads(
    (ROOT / "results" / "sky130_ci_manifest.json").read_text()
)
cal = json.loads(
    (ROOT / "results" / "pdk_calibration_analysis.json").read_text()
)
release = json.loads((ROOT / "release_requirements.json").read_text())
print("Project root:", ROOT)
print("PDK:", manifest["pdk_sources"])
print("Corners:", manifest["corners"])
print("Retained temperatures:", manifest["temperature_c"])


## Analytical PTAT expectation

The analytical model provides intuition and a cross-check, not a replacement for
transistor-level evidence.

In [ ]:
import sys

sys.path.insert(0, str(ROOT / "src"))
from ptat_model import PTATParams, nominal_slope_v_per_k

p = PTATParams(current_density_ratio=design["current_density_ratio"])
slope_uv_per_k = 1e6 * nominal_slope_v_per_k(p)
print(f"Analytical default slope: {slope_uv_per_k:.3f} uV/K")
for corner in ("tt", "ff", "ss"):
    slope = manifest["summary"]["mirror"][corner]["ptat_slope_uv_per_k"]
    print(f"Retained mirror {corner.upper()}: {slope:.3f} uV/K")

## Retained real-PDK temperature response

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for corner in ("tt", "ff", "ss"):
    csv_path = ROOT.joinpath(
        "results", "sky130_ci", f"ptat_mirror_{corner}.csv"
    )
    df = pd.read_csv(csv_path, skipinitialspace=True)
    ax.plot(
        df["temp_c"],
        1e3 * df["dvgs_v"],
        marker="o",
        label=corner.upper(),
    )
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("ΔVGS (mV)")
ax.set_title("Retained SKY130 PMOS-mirror PTAT response")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()


## Calibration result

Two-point endpoint calibration misses the internal target. The five-point PWL
architecture is therefore the release method rather than hiding the miss.

In [ ]:
two = cal["two_point_endpoint_calibration"]["worst_max_abs_error_c"]
pwl = cal["five_point_pwl_grid_calibration"]["worst_max_abs_error_c"]
anchors = cal["five_point_pwl_grid_calibration"]["anchors_c"]
print(f"Two-point worst retained sampled error: {two:.3f} °C")
print(f"Five-point PWL anchors: {anchors}")
print(f"Five-point PWL worst retained sampled error: {pwl:.3f} °C")
assert two > 0.5
pwl_target = release["release_targets"][
    "sampled_grid_pwl_max_abs_error_c_max"
]
assert pwl <= pwl_target

## Calibration error by retained dataset

In [ ]:
rows = []
per_dataset = cal["five_point_pwl_grid_calibration"]["per_dataset"]
for name, metrics in per_dataset.items():
    rows.append(
        {
            "dataset": name,
            "max_abs_error_c": metrics["max_abs_error_c"],
            "rms_error_c": metrics["rms_error_c"],
        }
    )
pd.DataFrame(rows).sort_values("max_abs_error_c", ascending=False)

## Behavioral ADC/readout budget

The project does not claim a transistor-level ADC. The release model uses a
12-bit, 1.8-V ADC with analog gain 10, selected to preserve headroom and keep the
quantized retained-grid calibration within 0.5 °C.

In [ ]:
import importlib.util

readout_path = ROOT / "readout_budget.py"
spec = importlib.util.spec_from_file_location("readout_budget", readout_path)
rb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rb)
readout = rb.analyze()
summary = {
    "status": readout["status"],
    "gain": readout["analog_gain"],
    "worst_quantization_rms_c": readout["worst_quantization_rms_c"],
    "worst_full_scale_utilization": readout[
        "worst_full_scale_utilization"
    ],
    "worst_quantized_pwl_sampled_error_c": readout[
        "worst_quantized_pwl_sampled_error_c"
    ],
}
print(json.dumps(summary, indent=2))
assert readout["status"] == "PASS"

## Power and mismatch boundaries

The retained PMOS-mirror testbench includes the PTAT sensing branches and PMOS
bias network, but the reference current source is ideal and the ADC is
behavioral. Therefore this project reports **core/testbench power only**, not
whole-chip power.

The retained TT/FF/SS branch-current mismatch (worst **0.372%**) is deterministic
corner behavior. A separate real SKY130 `tt_mm` Monte Carlo run used 100 seeds
and a 5 °C temperature grid. With the release five-point calibration, error
yield at ≤0.5 °C was **66%**, and branch-mismatch yield at ≤1% was **4%**.
Those results fail the internal 95% statistical targets and are retained as a
negative engineering result rather than hidden.


In [ ]:
summary = manifest["summary"]["mirror"]
print(
    "Worst retained mirror power (uW):",
    max(v["max_power_uw"] for v in summary.values()),
)
print(
    "Worst retained deterministic branch mismatch (%):",
    max(v["max_branch_mismatch_percent"] for v in summary.values()),
)

mismatch_path = ROOT.joinpath(
    "results", "full_pdk_20260926", "mismatch_summary.json"
)
mismatch = json.loads(mismatch_path.read_text())
print(
    "100-seed mismatch error yield <=0.5 C (%):",
    mismatch["yield_percent_error_le_target"],
)
print(
    "100-seed branch mismatch yield <=1% (%):",
    mismatch["yield_percent_branch_mismatch_le_1pct"],
)
print("Mismatch p95 max error (C):", mismatch["max_abs_error_c"]["p95"])
print("Mismatch worst max error (C):", mismatch["max_abs_error_c"]["worst"])

holdout_path = ROOT.joinpath(
    "results", "full_pdk_20260926", "mismatch_calibration_holdout.json"
)
holdout = json.loads(holdout_path.read_text())
print(
    "Six-point split-sample holdout yield <=0.5 C (%):",
    holdout["combined_point_estimate_for_shared_schedule"]["yield_percent"],
)
print(
    "Six-point shared anchors (C):",
    holdout["fold_a_selected_anchors_c"],
)


## Dense-grid and physical verification

The original retained run has eight temperature samples. A new pinned-PDK
dense run now adds **34 actual samples per dataset** on a 5 °C grid from −40 to
125 °C across TT/FF/SS. The fixed five-point PWL calibration reaches a worst
sampled-grid error of **0.472 °C**, so the deterministic dense-grid target passes.

The same full-PDK workflow completed a 100-seed local-mismatch run. That run
fails the current statistical targets, so it is evidence for the next design
iteration rather than a release-performance claim. A retrospective two-fold
split-sample study selected the same six-point anchor schedule in both folds:
**[-40, -25, -5, 25, 65, 125] °C**. Each 50-seed holdout fold reached **98%**
error yield at ≤0.5 °C, and the combined point estimate was also **98%**.
This reduces calibration-tuning leakage, but both folds still come from the
same retained 100-seed Monte Carlo run; it is therefore not treated as a new
independent Monte Carlo validation or as evidence that branch-current matching
meets its target.

A final layout is encouraged by Code-a-Chip but not required. The layout
directory defines the intended LVS boundary and evidence contract. DRC/LVS/PEX
remain **not claimed** until retained artifacts exist.


## Independent candidate validation

A completed full-PDK CI validation now provides a disjoint-seed check of the two main exploratory candidates. This evidence is **not** promoted into the release architecture because the statistical mismatch contract is not met.

- **1.2 V / 10 nA candidate, seeds 7001–7100 (100 samples):** 100% temperature-error yield at ≤0.5 °C, but only **5%** branch-mismatch yield at ≤1%; overall mismatch status **FAIL**.
- **Sizing candidate `i1_m16_s1`, seeds 9001–9100 (100 independent samples):** **66%** temperature-error yield and **64%** branch-mismatch yield; overall mismatch status **FAIL**.
- The sizing candidate separately passes deterministic dense TT/FF/SS verification (worst five-point PWL error **0.472 °C**, worst deterministic mirror mismatch **0.950%**) and the behavioral 12-bit / gain-10 readout budget (worst quantized sampled error **0.478 °C**).

The machine-evaluated qualification result is therefore **`NOT_QUALIFIED_FOR_RELEASE_REVIEW`**. This is a useful negative design-closure result: deterministic corners and readout are adequate, while genuinely independent local-mismatch statistics remain the limiting criterion. The release five-point calibration and nominal architecture are therefore left unchanged.

The compact retained record is `results/full_pdk_20260926/candidate_validation_summary.json` and records workflow run 36259524511, artifact provenance, disjoint seed ranges, and the final qualification components.


## Reproducibility

From this project directory run:

    python evidence_audit.py
    python pdk_calibration_analysis.py --check
    python readout_budget.py --check
    python submission_preflight.py

With ngspice and the pinned SKY130/open_pdks PDK installed:

    python run_sky130.py --mode both --corners tt ff ss --temps=-40:125:5
    python dense_characterization.py
    python mismatch_mc.py --samples 100 --temps=-40:125:5

The notebook remains reproducible without a local PDK installation because the
original run-221 CSV evidence and provenance are retained in the repository.

## References

1. IEEE SSCS Open-Source Ecosystem, **Code-a-Chip / ISSCC 2027** submission repository and rules: https://github.com/sscs-ose/sscs-ose-code-a-chip.github.io
2. OpenFASoC, **Temperature Sensor Generator** notebook/example: https://github.com/idea-fasoc/OpenFASOC/tree/main/docs/source/notebooks/temp-sense-gen
3. ngspice project documentation: https://ngspice.sourceforge.io/
4. SKY130 open PDK ecosystem / open_pdks project: https://github.com/RTimothyEdwards/open_pdks

No numerical result in this notebook is copied from these references; project results are generated from the analytical model or the retained repository evidence identified above.


## Limitations

- No silicon measurements are available.
- No post-layout result is claimed.
- The deterministic dense-grid five-point calibration passes the internal
  0.5 °C target, but the 100-seed `tt_mm` mismatch run does not meet the
  internal statistical targets: 66% error yield and 4% branch-mismatch yield.
- The six-point schedule is supported by a two-fold split-sample robustness
  study with 98% holdout yield in each fold, but both folds come from the same
  retained 100-seed run. It remains outside the release architecture until an
  independent Monte Carlo run confirms it.
- All reported temperature-error values are simulation results on explicit
  sampled grids, not guarantees between simulated temperatures.

These limitations are part of the result, not hidden exceptions.
